In [20]:
import pandas as pd
import duckdb
import polars as pl
import numpy as np
import time
from pathlib import Path

print("Pandas:", pd.__version__)
print("DuckDB:", duckdb.__version__)
print("Polars:", pl.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.6
DuckDB: 1.5.6
Polars: 1.44.2
NumPy: 2.5.3


In [21]:
# Create a reproducible 100,000-row dataset

np.random.seed(42)

n = 100_000

categories = ["Clothing", "Electronics", "Furniture"]
states = [
    "Karnataka", "Maharashtra", "Delhi", "Tamil Nadu",
    "Kerala", "Gujarat", "Rajasthan", "West Bengal"
]
cities = [
    "Bengaluru", "Mumbai", "Delhi", "Chennai",
    "Kochi", "Ahmedabad", "Jaipur", "Kolkata"
]

df = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "order_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2018-01-01", "2019-12-31"),
            size=n
        )
    ),
    "category": np.random.choice(categories, size=n),
    "state": np.random.choice(states, size=n),
    "city": np.random.choice(cities, size=n),
    "quantity": np.random.randint(1, 15, size=n),
    "amount": np.round(np.random.uniform(100, 2000, size=n), 2),
    "profit": np.round(np.random.uniform(-100, 300, size=n), 2)
})

csv_path = Path("Week 5/duckdb_100k.csv")
df.to_csv(csv_path, index=False)

print("Dataset created successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Saved to:", csv_path)

OSError: Cannot save file into a non-existent directory: 'Week 5'

In [ ]:
Path("Week 5/duckdb_100k.csv")

WindowsPath('Week 5/duckdb_100k.csv')

In [ ]:
from pathlib import Path

print("Current folder:")
print(Path.cwd())

print("\nCSV exists:")
print(Path("duckdb_100k.csv").exists())

print("\nCSV path:")
print(Path("duckdb_100k.csv").resolve())

Current folder:
c:\Users\DELL\Desktop\Cynaris-Internship\Week 5

CSV exists:
False

CSV path:
C:\Users\DELL\Desktop\Cynaris-Internship\Week 5\duckdb_100k.csv


In [ ]:
# Create a reproducible 100,000-row dataset

np.random.seed(42)

n = 100_000

categories = ["Clothing", "Electronics", "Furniture"]

states = [
    "Karnataka", "Maharashtra", "Delhi", "Tamil Nadu",
    "Kerala", "Gujarat", "Rajasthan", "West Bengal"
]

cities = [
    "Bengaluru", "Mumbai", "Delhi", "Chennai",
    "Kochi", "Ahmedabad", "Jaipur", "Kolkata"
]

df = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "order_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2018-01-01", "2019-12-31"),
            size=n
        )
    ),
    "category": np.random.choice(categories, size=n),
    "state": np.random.choice(states, size=n),
    "city": np.random.choice(cities, size=n),
    "quantity": np.random.randint(1, 15, size=n),
    "amount": np.round(np.random.uniform(100, 2000, size=n), 2),
    "profit": np.round(np.random.uniform(-100, 300, size=n), 2)
})

# Save in the current Week 5 folder
csv_path = Path("duckdb_100k.csv")
df.to_csv(csv_path, index=False)

print("Dataset created successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Saved to:", csv_path.resolve())

Dataset created successfully!
Rows: 100000
Columns: 8
Saved to: C:\Users\DELL\Desktop\Cynaris-Internship\Week 5\duckdb_100k.csv


In [ ]:
print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

display(df.head())

Shape: (100000, 8)
Missing values: 0
Duplicate rows: 0


,order_id,order_date,category,state,city,quantity,amount,profit
0,1,2018-04-13,Clothing,Delhi,Ahmedabad,9,648.72,-46.05
1,2,2019-03-12,Clothing,Rajasthan,Ahmedabad,12,619.08,-28.12
2,3,2018-09-28,Electronics,Kerala,Chennai,10,1076.51,-4.91
3,4,2018-04-17,Furniture,Delhi,Ahmedabad,13,884.38,263.14
4,5,2018-03-13,Furniture,Gujarat,Jaipur,2,1997.23,82.23


In [ ]:
query1 = """
SELECT
    category,
    ROUND(SUM(amount), 2) AS total_amount
FROM orders
GROUP BY category
ORDER BY total_amount DESC
"""

duckdb_q1 = con.execute(query1).df()

display(duckdb_q1)

NameError: name 'con' is not defined

In [ ]:
import duckdb

con = duckdb.connect()

con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT *
    FROM read_csv_auto('duckdb_100k.csv')
""")

row_count = con.execute("""
    SELECT COUNT(*) FROM orders
""").fetchone()[0]

print("DuckDB table created successfully!")
print("Rows loaded:", row_count)

DuckDB table created successfully!
Rows loaded: 100000


In [ ]:
query1 = """
SELECT
    category,
    ROUND(SUM(amount), 2) AS total_amount
FROM orders
GROUP BY category
ORDER BY total_amount DESC
"""

duckdb_q1 = con.execute(query1).df()

display(duckdb_q1)

,category,total_amount
0,Clothing,35117283.70
1,Electronics,34918344.66
2,Furniture,34832551.35


In [ ]:
query2 = """
SELECT
    state,
    ROUND(AVG(profit), 2) AS average_profit
FROM orders
GROUP BY state
ORDER BY average_profit DESC
"""

duckdb_q2 = con.execute(query2).df()

display(duckdb_q2)

,state,average_profit
0,Karnataka,101.01
1,Rajasthan,100.87
2,Gujarat,100.63
3,Delhi,100.13
4,Tamil Nadu,100.09
5,Kerala,99.94
6,West Bengal,99.50
7,Maharashtra,99.06


In [ ]:
query3 = """
SELECT
    city,
    SUM(quantity) AS total_quantity
FROM orders
GROUP BY city
ORDER BY total_quantity DESC
LIMIT 10
"""

duckdb_q3 = con.execute(query3).df()

display(duckdb_q3)

,city,total_quantity
0,Kolkata,95653.0
1,Bengaluru,95249.0
2,Delhi,94196.0
3,Jaipur,93803.0
4,Chennai,93630.0
5,Kochi,92596.0
6,Mumbai,92248.0
7,Ahmedabad,91611.0


In [ ]:
query4 = """
SELECT
    EXTRACT(MONTH FROM order_date) AS month,
    ROUND(SUM(amount), 2) AS total_amount
FROM orders
GROUP BY month
ORDER BY month
"""

duckdb_q4 = con.execute(query4).df()

display(duckdb_q4)

,month,total_amount
0,1,8962144.64
1,2,8070841.60
2,3,8808043.34
3,4,8548056.83
4,5,8902924.73
5,6,8771463.34
6,7,8831767.32
7,8,9222672.50
8,9,8571053.72
9,10,8909069.30


In [ ]:
query5 = """
SELECT
    category,
    COUNT(*) AS order_count,
    ROUND(SUM(profit), 2) AS total_profit
FROM orders
WHERE amount > 500
GROUP BY category
ORDER BY total_profit DESC
"""

duckdb_q5 = con.execute(query5).df()

display(duckdb_q5)

,category,order_count,total_profit
0,Electronics,26309,2662335.20
1,Clothing,26418,2643417.42
2,Furniture,26227,2632663.18


In [ ]:
# Load the same CSV into Pandas
pandas_start = time.perf_counter()

pandas_df = pd.read_csv("duckdb_100k.csv")

# Query 1
pandas_q1 = (
    pandas_df.groupby("category", as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "total_amount"})
    .sort_values("total_amount", ascending=False)
)

# Query 2
pandas_q2 = (
    pandas_df.groupby("state", as_index=False)["profit"]
    .mean()
    .rename(columns={"profit": "average_profit"})
    .sort_values("average_profit", ascending=False)
)

# Query 3
pandas_q3 = (
    pandas_df.groupby("city", as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "total_quantity"})
    .sort_values("total_quantity", ascending=False)
    .head(10)
)

# Query 4
pandas_df["month"] = pd.to_datetime(pandas_df["order_date"]).dt.month

pandas_q4 = (
    pandas_df.groupby("month", as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "total_amount"})
    .sort_values("month")
)

# Query 5
pandas_q5 = (
    pandas_df[pandas_df["amount"] > 500]
    .groupby("category")
    .agg(
        order_count=("category", "size"),
        total_profit=("profit", "sum")
    )
    .reset_index()
    .sort_values("total_profit", ascending=False)
)

pandas_time = time.perf_counter() - pandas_start

print(f"Pandas total time: {pandas_time:.4f} seconds")

Pandas total time: 2.5586 seconds


In [ ]:
# Load the same CSV into Polars
polars_start = time.perf_counter()

polars_df = pl.read_csv("duckdb_100k.csv")

# Query 1
polars_q1 = (
    polars_df
    .group_by("category")
    .agg(pl.col("amount").sum().alias("total_amount"))
    .sort("total_amount", descending=True)
)

# Query 2
polars_q2 = (
    polars_df
    .group_by("state")
    .agg(pl.col("profit").mean().alias("average_profit"))
    .sort("average_profit", descending=True)
)

# Query 3
polars_q3 = (
    polars_df
    .group_by("city")
    .agg(pl.col("quantity").sum().alias("total_quantity"))
    .sort("total_quantity", descending=True)
    .head(10)
)

# Query 4
polars_q4 = (
    polars_df
    .with_columns(
        pl.col("order_date").str.to_date().dt.month().alias("month")
    )
    .group_by("month")
    .agg(pl.col("amount").sum().alias("total_amount"))
    .sort("month")
)

# Query 5
polars_q5 = (
    polars_df
    .filter(pl.col("amount") > 500)
    .group_by("category")
    .agg([
        pl.len().alias("order_count"),
        pl.col("profit").sum().alias("total_profit")
    ])
    .sort("total_profit", descending=True)
)

polars_time = time.perf_counter() - polars_start

print(f"Polars total time: {polars_time:.4f} seconds")

Polars total time: 1.6179 seconds


In [ ]:
# Fresh DuckDB connection for a fair benchmark

duckdb_start = time.perf_counter()

benchmark_con = duckdb.connect()

benchmark_con.execute("""
    CREATE OR REPLACE TABLE orders_benchmark AS
    SELECT *
    FROM read_csv_auto('duckdb_100k.csv')
""")

# Query 1
benchmark_con.execute("""
    SELECT
        category,
        ROUND(SUM(amount), 2) AS total_amount
    FROM orders_benchmark
    GROUP BY category
    ORDER BY total_amount DESC
""").df()

# Query 2
benchmark_con.execute("""
    SELECT
        state,
        ROUND(AVG(profit), 2) AS average_profit
    FROM orders_benchmark
    GROUP BY state
    ORDER BY average_profit DESC
""").df()

# Query 3
benchmark_con.execute("""
    SELECT
        city,
        SUM(quantity) AS total_quantity
    FROM orders_benchmark
    GROUP BY city
    ORDER BY total_quantity DESC
    LIMIT 10
""").df()

# Query 4
benchmark_con.execute("""
    SELECT
        EXTRACT(MONTH FROM order_date) AS month,
        ROUND(SUM(amount), 2) AS total_amount
    FROM orders_benchmark
    GROUP BY month
    ORDER BY month
""").df()

# Query 5
benchmark_con.execute("""
    SELECT
        category,
        COUNT(*) AS order_count,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders_benchmark
    WHERE amount > 500
    GROUP BY category
    ORDER BY total_profit DESC
""").df()

duckdb_time = time.perf_counter() - duckdb_start

print(f"DuckDB total time: {duckdb_time:.4f} seconds")

DuckDB total time: 1.3494 seconds


In [ ]:
# Verify that Pandas and Polars produce the same results as DuckDB

print("Pandas vs DuckDB")
print("Q1:", pandas_q1)
print("Q2:", pandas_q2)
print("Q3:", pandas_q3)
print("Q4:", pandas_q4)
print("Q5:", pandas_q5)

print("\nPolars results")
print("Q1:", polars_q1)
print("Q2:", polars_q2)
print("Q3:", polars_q3)
print("Q4:", polars_q4)
print("Q5:", polars_q5)

Pandas vs DuckDB
Q1:       category  total_amount
0     Clothing   35117283.70
1  Electronics   34918344.66
2    Furniture   34832551.35
Q2:          state  average_profit
2    Karnataka      101.008382
5    Rajasthan      100.870858
1      Gujarat      100.631875
0        Delhi      100.134209
6   Tamil Nadu      100.089724
3       Kerala       99.939816
7  West Bengal       99.501430
4  Maharashtra       99.061579
Q3:         city  total_quantity
6    Kolkata           95653
1  Bengaluru           95249
3      Delhi           94196
4     Jaipur           93803
2    Chennai           93630
5      Kochi           92596
7     Mumbai           92248
0  Ahmedabad           91611
Q4:     month  total_amount
0       1    8962144.64
1       2    8070841.60
2       3    8808043.34
3       4    8548056.83
4       5    8902924.73
5       6    8771463.34
6       7    8831767.32
7       8    9222672.50
8       9    8571053.72
9      10    8909069.30
10     11    8442264.43
11     12    8827877.96

In [ ]:
print("W5D3 BENCHMARK RESULTS")
print("-----------------------")
print(f"Pandas : {pandas_time:.4f} seconds")
print(f"Polars : {polars_time:.4f} seconds")
print(f"DuckDB : {duckdb_time:.4f} seconds")

print("\nFastest tool:", min(
    [("Pandas", pandas_time),
     ("Polars", polars_time),
     ("DuckDB", duckdb_time)],
    key=lambda x: x[1]
)[0])

W5D3 BENCHMARK RESULTS
-----------------------
Pandas : 2.5586 seconds
Polars : 1.6179 seconds
DuckDB : 1.3494 seconds

Fastest tool: DuckDB


# W5D3 – DuckDB vs Pandas vs Polars

## Objective

Compare DuckDB, Pandas, and Polars for analytical operations on a 100,000-row CSV dataset.

The benchmark dataset is synthetic and was generated reproducibly using NumPy. It contains 100,000 orders and 8 columns: order ID, order date, category, state, city, quantity, amount, and profit.

## Dataset Validation

- Rows: 100,000
- Columns: 8
- Missing values: 0
- Duplicate rows: 0

## Analytical Queries

Five analytical operations were performed:

1. Total sales amount by category.
2. Average profit by state.
3. Total quantity by city.
4. Total sales amount by month.
5. Order count and total profit by category for orders with amount greater than 500.

The results produced by DuckDB, Pandas, and Polars were consistent for the five analytical queries.

## Benchmark Results

| Tool | Total Time |
|---|---:|
| Pandas | 2.5586 seconds |
| Polars | 1.6179 seconds |
| DuckDB | 1.3494 seconds |

For this dataset, machine, and workload, DuckDB recorded the lowest execution time, followed by Polars and Pandas. These results are workload-specific and should not be interpreted as a universal performance ranking.

## Pandas vs DuckDB vs Polars

| Criteria | Pandas | DuckDB | Polars |
|---|---|---|---|
| Query syntax | Python DataFrame API | SQL | DataFrame expression API |
| Execution approach | Eager in-memory processing | Vectorized analytical SQL engine | Columnar, parallel DataFrame engine |
| File analytics | Loads data into DataFrame | Can query files directly using SQL | Reads files into DataFrame/LazyFrame |
| Best use case | General data manipulation and EDA | Local analytical SQL and large-file analytics | High-performance DataFrame workflows |
| Ease of learning | Familiar for Python users | Easy for users familiar with SQL | Requires learning Polars expressions |

## Key Findings

- DuckDB performed fastest in this benchmark at 1.3494 seconds.
- Polars completed the same benchmark in 1.6179 seconds.
- Pandas completed the benchmark in 2.5586 seconds.
- All three tools produced consistent analytical results for the five queries.
- DuckDB is useful when analysts prefer SQL for analytical workloads and need to work directly with local files.
- Polars is useful for high-performance DataFrame processing.
- Pandas remains useful for general-purpose Python data analysis and exploratory data work.

## Business Recommendation

For SQL-based analytical workloads involving large local files, DuckDB is a strong option because it allows analysts to use familiar SQL without requiring a separate database server. Polars is useful when a high-performance Python DataFrame workflow is preferred, while Pandas remains suitable for general data analysis and EDA.

The benchmark should be repeated with larger datasets and representative production workloads before making a tool-selection decision for a production environment.

## Limitations

- The dataset is synthetic and contains 100,000 rows.
- The benchmark was performed on one machine.
- Execution time can vary depending on hardware, software versions, caching, and workload.
- The benchmark does not represent all possible Pandas,